# IBKR HYG 长历史来源单次探针

冻结 ARCA HYG 精确 contract_id 的只读日线 4000 bar 请求；完整保存连接器响应，NAS 回读。若连接器上限拒绝则止步，不推断其他来源不存在。

In [1]:
import json,hashlib,tarfile,tempfile,subprocess,shlex
from pathlib import Path
from datetime import datetime,timezone
from src.data.nas_market_data import _settings,nas_api_call
root=Path.cwd();card=root/'research/stage1_ibkr_credit_etf_source_probe_20261023';plan=json.loads((card/'plan.json').read_text());response_path=root/'data/snapshots/stage1_ibkr_credit_etf_source_probe_20261023/response.json';payload=json.loads(response_path.read_text())
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen']
assert payload['request']['contract_id']==plan['request']['contract_id'] and payload['request']['step_count']==plan['request']['step_count'] and payload['response']=={'error':'Step count more than 1000 is not allowed'}
assert not (card/'attempt_started.json').exists()
(card/'attempt_started.json').write_text(json.dumps({'status':'one_read_only_IBKR_history_request_done','observed_at_utc':datetime.now(timezone.utc).isoformat(),'requested_bars':4000,'returned_bars':0,'broker_writes':0},indent=2)+'\n')
manifest={'plan_sha256':sha(card/'plan.json'),'connector_response_sha256':sha(response_path),'connector_response_bytes':response_path.stat().st_size,'request_count':1,'returned_price_bars':0};(card/'source_manifest.json').write_text(json.dumps(manifest,indent=2)+'\n')
settings=_settings(root)
with tempfile.TemporaryDirectory(prefix='hyg-ibkr-source-') as temp:
 archive=Path(temp)/'source.tar.gz'
 with tarfile.open(archive,'w:gz') as out:
  out.add(response_path,arcname='raw/response.json');out.add(card/'source_manifest.json',arcname='source_manifest.json')
 digest=sha(archive);evidence=nas_api_call(origin=settings['origin'],token=settings['token'],method='PUT',path='/api/research/artifacts/'+digest,upload=archive,executor_id='soxl-minute-research')['artifact']
 assert evidence['sha256']==digest and evidence['size_bytes']==archive.stat().st_size
 remote=settings['artifact_root'].rstrip('/')+'/'+digest[:2]+'/'+digest[2:4]+'/'+digest
 with tempfile.TemporaryDirectory(prefix='hyg-ibkr-restore-') as rd:
  restored=Path(rd)/'restore.tar.gz'
  with restored.open('wb') as out:run=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(remote)],stdout=out,stderr=subprocess.PIPE,timeout=90)
  assert run.returncode==0 and sha(restored)==digest
  with tarfile.open(restored,'r:gz') as source:
   member=source.extractfile('raw/response.json');assert member and hashlib.sha256(member.read()).hexdigest()==manifest['connector_response_sha256']
result={'status':'IBKR_history_4000_daily_bar_request_rejected_step_count_max1000_source_only','plan_sha256':manifest['plan_sha256'],'connector_response_sha256':manifest['connector_response_sha256'],'requested_daily_bars':4000,'returned_bars':0,'error':payload['response']['error'],'NAS_archive_sha256':digest,'NAS_remote_and_scoped_restore_verified':True,'HYG_2011_source_qualified':False,'LQD_request_attempted':False,'market_outcome_or_SOXX_reads':0,'model_fits':0,'native_accounts':0,'broker_writes':0}
(card/'source_result.json').write_text(json.dumps(result,indent=2)+'\n')
print(result)

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{'status': 'IBKR_history_4000_daily_bar_request_rejected_step_count_max1000_source_only', 'plan_sha256': '321cd90a06587264ee852e1133813b45130115c143abdf43025b70287bfb5285', 'connector_response_sha256': '4b5c3cc5736e400ea7fbb3d21b2c4cdc1ad4902026eb57358ad0dcfacbe14672', 'requested_daily_bars': 4000, 'returned_bars': 0, 'error': 'Step count more than 1000 is not allowed', 'NAS_archive_sha256': '8ab80397505f25daf0781d88000215542535a7e359415e9bca43a4f45973a98a', 'NAS_remote_and_scoped_restore_verified': True, 'HYG_2011_source_qualified': False, 'LQD_request_attempted': False, 'market_outcome_or_SOXX_reads': 0, 'model_fits': 0, 'native_accounts': 0, 'broker_writes': 0}


## 中文结论

美股 ARCA HYG 合同 ID 43652089 的固定只读 4000 根日线请求被 IBKR 连接器以 `Step count more than 1000 is not allowed` 拒绝，0 价格柱；本接口无历史结束日期参数。无二次请求或 LQD 请求。连接器响应已 NAS 原字节 SHA-256 `8ab80397505f25daf0781d88000215542535a7e359415e9bca43a4f45973a98a` 回读。HYG 2011 来源未合格，0 SOXX 结果、0 拟合、0 账户、0 券商写。